# Stage 1: Generate the fintech A/B test data

Run each cell top to bottom. Output: `data/users.csv` and `data/experiment_assignments.csv`.

In [1]:
import random
from datetime import date, timedelta
from pathlib import Path
import pandas as pd

random.seed(42)

number_of_users = 10000

data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

### Cell 2: users table

In [2]:
start_date = date(2026, 1, 1)

channels = ["organic", "paid_social", "referral", "search_ads", "partner"]
channel_weights = [30, 25, 15, 20, 10]

devices = ["ios", "android", "web"]
device_weights = [45, 40, 15]

age_bands = ["18-24", "25-34", "35-44", "45-54", "55+"]
age_weights = [22, 34, 22, 14, 8]

states = ["IL", "NY", "CA", "TX", "FL", "OTHER"]
state_weights = [18, 15, 17, 12, 10, 28]

rows = []

for i in range(1, number_of_users + 1):
    user_id = "U" + str(i).zfill(5)

    days_after_start = random.randint(0, 180)
    signup_date = start_date + timedelta(days=days_after_start)

    channel = random.choices(channels, weights=channel_weights)[0]
    device = random.choices(devices, weights=device_weights)[0]
    age_band = random.choices(age_bands, weights=age_weights)[0]
    state = random.choices(states, weights=state_weights)[0]

    row = {
        "user_id": user_id,
        "signup_date": signup_date,
        "acquisition_channel": channel,
        "device": device,
        "age_band": age_band,
        "state": state,
    }
    rows.append(row)

users = pd.DataFrame(rows)

### Cell 3: experiment assignments

In [3]:
experiment_id = "EXP_001_onboarding_v2"

assignment_rows = []

for row in rows:
    variant = random.choice(["control", "treatment"])

    assignment = {
        "user_id": row["user_id"],
        "experiment_id": experiment_id,
        "variant": variant,
        "assigned_at": row["signup_date"],
    }
    assignment_rows.append(assignment)

assignments = pd.DataFrame(assignment_rows)

### Cell 4: validate + save

In [4]:
assert users["user_id"].is_unique
assert len(users) == len(assignments) == number_of_users
assert users.isna().sum().sum() == 0

users.to_csv(data_folder / "users.csv", index=False)
assignments.to_csv(data_folder / "experiment_assignments.csv", index=False)

### Check your work

In [5]:
users.head()

,user_id,signup_date,acquisition_channel,device,age_band,state
0,U00001,2026-06-13,organic,android,25-34,IL
1,U00002,2026-01-27,referral,web,18-24,CA
2,U00003,2026-01-08,organic,ios,35-44,TX
3,U00004,2026-06-16,search_ads,ios,25-34,NY
4,U00005,2026-01-02,search_ads,ios,25-34,NY


In [6]:
assignments['variant'].value_counts()

variant
treatment    5009
control      4991
Name: count, dtype: int64